# Poker NLHE — agente NFSP che impara da solo (RLCard) + export della range chart

Questo notebook addestra un agente di **reinforcement learning (NFSP)** a giocare al
**No-Limit Texas Hold'em** giocando contro se stesso (self-play), usando la libreria
**RLCard**. Alla fine **estrae una range chart** in stile solver dalle decisioni preflop
dell'agente, come la 'SB Open' della figura di riferimento.

**Onesta intellettuale:** partendo da zero su Colab gratis NON si raggiunge il livello GTO
del Hold'em completo (servirebbe molto piu' calcolo). L'obiettivo qui e' mostrare la
**pipeline** completa: ambiente reale a 4 giri di puntata -> apprendimento -> chart appresa.
Aumenta `NUM_EPISODES` per risultati migliori; con una GPU (Runtime > Cambia tipo) va piu' veloce.


## 1. Installazione


In [ ]:
!pip install 'rlcard[torch]' matplotlib -q


## 2. Ambiente e agente NFSP


In [ ]:
import torch, rlcard
from rlcard.agents import NFSPAgent, RandomAgent
from rlcard.utils import set_seed, tournament, reorganize, Logger

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
set_seed(42)

env = rlcard.make('no-limit-holdem', config={'seed': 42})
print('azioni:', env.num_actions, '| shape stato:', env.state_shape[0])

agent = NFSPAgent(
    num_actions=env.num_actions,
    state_shape=env.state_shape[0],
    hidden_layers_sizes=[128, 128],
    q_mlp_layers=[128, 128],
    device=device,
)
# self-play: lo stesso agente gioca su entrambe le posizioni
env.set_agents([agent, agent])


## 3. Addestramento (self-play)


In [ ]:
NUM_EPISODES = 20000   # alza a 100k+ per risultati migliori
EVAL_EVERY   = 2000

eval_env = rlcard.make('no-limit-holdem', config={'seed': 0})
eval_env.set_agents([agent, RandomAgent(num_actions=env.num_actions)])

for episode in range(NUM_EPISODES):
    agent.sample_episode_policy()                 # alterna best-response / media
    trajectories, payoffs = env.run(is_training=True)
    trajectories = reorganize(trajectories, payoffs)
    for player_traj in trajectories:              # alimenta entrambi i giocatori
        for ts in player_traj:
            agent.feed(ts)
    if episode % EVAL_EVERY == 0:
        reward = tournament(eval_env, 2000)[0]     # BB medie vs RandomAgent
        print(f'episodio {episode:>6}  reward vs random = {reward:+.3f}')
print('Fatto.')


## 4. Range chart renderer (incorporato)
Stessa logica del file `range_chart.py`: disegna la griglia 13x13 con le frequenze.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
RANKS='AKQJT98765432'; IDX={r:i for i,r in enumerate(RANKS)}
def combos(r,c): return 6 if r==c else (4 if r<c else 12)
def label(r,c):
    hi,lo=RANKS[min(r,c)],RANKS[max(r,c)]; return hi+hi if r==c else hi+lo
def fcol(f):
    red=(0.85,0.16,0.16); return tuple(1+f*(red[i]-1) for i in range(3))
def draw_freq(freq, title='SB Open (appreso)', fname='appreso.png'):
    # freq: dizionario {(riga,colonna): frequenza_raise}
    played=sum(combos(r,c)*f for (r,c),f in freq.items()); pct=100*played/1326
    fig,ax=plt.subplots(figsize=(7.2,7.6))
    for r in range(13):
        for c in range(13):
            f=freq.get((r,c),0.0)
            ax.add_patch(Rectangle((c,12-r),1,1,facecolor=fcol(f),edgecolor='black',lw=1))
            ax.text(c+.5,12-r+.5,label(r,c),ha='center',va='center',fontsize=8,fontweight='bold',color='#1a1a2e')
    ax.set_xlim(0,13); ax.set_ylim(-.5,14); ax.axis('off'); ax.set_aspect('equal')
    ax.text(0,13.3,title,fontsize=18,fontweight='bold')
    ax.text(13,13.3,f'{pct:.1f}%',fontsize=18,fontweight='bold',color='#c92a2a',ha='right')
    plt.tight_layout(); plt.savefig(fname,dpi=130,bbox_inches='tight'); plt.show()


## 5. Estrazione della chart dalle decisioni preflop dell'agente
Facciamo giocare l'agente (modalita' valutazione) e registriamo, per ogni **mano iniziale**,
con che frequenza apre/rilancia. Quelle frequenze diventano la chart.


In [ ]:
import numpy as np
from collections import defaultdict

RMAP={'A':14,'K':13,'Q':12,'J':11,'T':10,'9':9,'8':8,'7':7,'6':6,'5':5,'4':4,'3':3,'2':2}
def cell_of(cards):
    # cards es. ['SA','HK'] -> posizione (riga,colonna) nella griglia 13x13
    s1,r1=cards[0][0],cards[0][1]; s2,r2=cards[1][0],cards[1][1]
    v1,v2=RMAP[r1],RMAP[r2]
    hi,lo=max(v1,v2),min(v1,v2)
    def gi(v): return IDX['AKQJT98765432'[14-v]]   # indice nell'ordine A,K,Q,...,2
    a,b=gi(hi),gi(lo)
    if hi==lo: return (a,a)
    suited = (s1==s2)
    return (a,b) if suited else (b,a)   # suited sopra diagonale, offsuit sotto

# quali ID-azione sono 'raise'? in NLHE di RLCard le azioni >=2 sono i rilanci/all-in
def is_raise(action_id): return action_id >= 2

counts=defaultdict(lambda:[0,0])  # cella -> [raise, totali]
N_HANDS=20000
agent.eval_mode = True if hasattr(agent,'eval_mode') else None
for _ in range(N_HANDS):
    state, pid = env.reset()
    raw=state['raw_obs']
    if len(raw.get('public_cards',[]))==0:        # siamo preflop
        cell=cell_of(raw['hand'])
        action,_=agent.eval_step(state)
        counts[cell][1]+=1
        if is_raise(action): counts[cell][0]+=1

freq={cell:(r/t if t else 0) for cell,(r,t) in counts.items() if t>=5}
draw_freq(freq, title='Apertura appresa (NFSP)', fname='appreso.png')
print('Celle osservate:', len(freq))


## Note
- Se `is_raise` non corrisponde alle azioni della tua versione di RLCard, stampa
  `env.actions` o `raw['legal_actions']` per vedere i nomi e adatta la soglia.
- Con pochi episodi la chart sara' 'rumorosa': e' normale. Aumenta `NUM_EPISODES`.
- Per confrontarla con l'ottimo teorico puoi affiancare un solver/holdem CFR su versioni
  ridotte (Leduc) come fatto nel progetto Kuhn.
